# Week 5: Spatial operations: joins, buffers, intersects, dissolve

**Track:** Orbital Analyst (Intermediate)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/5/](https://launchdetect.com/academy/week/5/)  
**Track index:** [https://launchdetect.com/academy/orbital-analyst/](https://launchdetect.com/academy/orbital-analyst/)

---

_The core verbs of spatial analysis. Spatial joins, buffers, intersections, and dissolve — taught with launch-site and airspace-exclusion-zone data._


## Why this week matters

Spatial analysis is where GIS becomes useful instead of just decorative. The five core verbs — spatial join, buffer, intersection, union, dissolve — show up in every single space-domain workflow you'll ever build, often multiple times in the same pipeline.

This week's specific scenario, a launch maritime exclusion zone, is a real operational artifact: before every launch from a coastal pad, the operator and national maritime authority publish a NOTMAR (Notice to Mariners) defining a polygon that vessels must avoid. The polygon is a geodesic buffer around the predicted downrange trajectory, intersected with shipping-lane density. Building it teaches the right verbs while producing a real-world-shaped output.


## Learning objectives

By the end of this lab you will be able to:

- Run a spatial join in QGIS and in geopandas
- Compute buffers and explain when geodesic vs planar matters
- Run intersection, union, and difference operations
- Use dissolve to aggregate features by attribute


## Setup — and why these dependencies

`pyproj.Geod` for *geodesic* (ellipsoid-accurate) operations. Critical: planar buffers and distances drift dramatically wrong at high latitudes and at large scales. Geodesic operations stay correct anywhere on Earth.

Optionally `geopandas` and `shapely` for batched operations over many features.


In [ ]:
# Install required packages
!pip install -q leafmap[common] skyfield geopandas shapely


## The approach (and why)

Build a 50 km geodesic exclusion buffer around Cape Canaveral by walking a circle of 36 points (every 10° of azimuth from 0° to 360°) at exactly 50 km geodesic distance from the pad. Connect them into a polygon.

**Why 36 points?** Smooth enough to look like a circle on a global map (any error is sub-100m), small enough to render fast. For finer work bump to 360 (one point per degree) — the cost is linear in point count.

**Why geodesic and not just `shapely.buffer(0.45)` (0.45 degrees ≈ 50 km)?** Because 1° of longitude at Cape Canaveral (28°N) is ~98 km, but at the equator it's 111 km, and at 60°N it's 55 km. A planar buffer would be elliptical, not circular, and the more you move north or south the more wrong it gets.


In [ ]:
# Week 5: maritime exclusion zone for a launch via geodesic buffer.
import leafmap.foliumap as leafmap
from pyproj import Geod

geod = Geod(ellps="WGS84")
pad_lat, pad_lon = 28.49, -80.60  # Cape Canaveral

# Build a 50 km geodesic buffer around the pad (rough launch exclusion)
buffer_pts = [geod.fwd(pad_lon, pad_lat, az, 50_000)[:2] for az in range(0, 361, 10)]

buffer_geojson = {
    "type": "Feature",
    "geometry": {"type": "Polygon", "coordinates": [[[x, y] for x, y in buffer_pts]]},
    "properties": {"name": "50 km exclusion zone"}
}

m = leafmap.Map(center=[pad_lat, pad_lon], zoom=8)
m.add_marker((pad_lat, pad_lon), popup="Cape Canaveral")
m.add_geojson(buffer_geojson, style={"color": "#ff6b35", "fillOpacity": 0.2})
m

# TODO: extend the buffer along a predicted trajectory (Week 8) instead of
# a circle, intersect with AIS shipping lanes, output the resulting polygon.


## What just happened — and why it works

The map shows the pad as a marker and the exclusion zone as a translucent orange polygon. The polygon is closed (the last point equals the first) because GeoJSON polygon rings must be closed.

Notice the polygon shape looks like a perfect circle on this small-area map but would look slightly elliptical on a very-large-area projection. The geodesic math preserves *real-world* distance (you really can walk 50 km in any direction and hit the ring), at the cost of visual non-circularity in some projections. Real distance > visual circularity, always.


## Common gotchas

- **Planar buffers are tempting and almost always wrong.** Use `pyproj.Geod.fwd()` or `geopy.distance.geodesic` for any real-distance calculation.
- **Polygon winding matters.** GeoJSON spec says exterior rings should be counter-clockwise (right-hand rule), interior rings (holes) clockwise. Not all consumers enforce this, but PostGIS does. If you get spurious 'invalid geometry' errors, winding is the usual culprit.
- **`from_crs(...).transform()` chains.** When you're chaining transforms across multiple CRSes, build a `Transformer` per pair and reuse it. Constructing one per call is slow.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/5/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
